In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to dataset files: /kaggle/input/gtzan-dataset-music-genre-classification


In [2]:
import os

file = os.listdir(path)
files_in_dir = os.listdir(path + '/' + file[0])

print(files_in_dir)

path_to_files = path + '/' + file[0]

['features_3_sec.csv', 'features_30_sec.csv', 'images_original', 'genres_original']


In [3]:
path_to_csv_3 =  path_to_files + '/' + files_in_dir[1]
path_to_csv_3

'/kaggle/input/gtzan-dataset-music-genre-classification/Data/features_30_sec.csv'

In [4]:
import pandas as pd
import numpy as np

path_to_csv_3 = os.path.join(path, "Data", "features_3_sec.csv")
df = pd.read_csv(path_to_csv_3)

df['time'] = df['filename'].str[-5].astype('int')
df = df.drop(['filename','length'], axis=1)

lab = {l: i for i,l in enumerate(np.unique(df['label']))}

df['res'] = df['label'].map(lab).astype('int')

df = df.drop(['label'], axis=1)

df

,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,rolloff_var,...,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,time,res
0,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,3714.560359,1.080790e+06,...,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,0,0
1,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,3869.682242,6.722448e+05,...,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,1,0
2,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,3997.639160,7.907127e+05,...,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,2,0
3,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,3568.300218,9.216524e+05,...,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,3,0
4,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,3469.992864,6.102111e+05,...,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,0.349126,0.080515,0.050019,0.000097,1499.083005,164266.886443,1718.707215,85931.574523,3015.559458,8.479527e+05,...,-9.094270,38.326839,-4.246976,31.049839,-5.625813,48.804092,1.818823,38.966969,5,9
9986,0.372564,0.082626,0.057897,0.000088,1847.965128,281054.935973,1906.468492,99727.037054,3746.694524,1.170890e+06,...,-12.375726,66.418587,-3.081278,54.414265,-11.960546,63.452255,0.428857,18.697033,6,9
9987,0.347481,0.089019,0.052403,0.000701,1346.157659,662956.246325,1561.859087,138762.841945,2442.362154,2.602871e+06,...,-2.524483,21.778994,4.809936,25.980829,1.775686,48.582378,-0.299545,41.586990,7,9
9988,0.387527,0.084815,0.066430,0.000320,2084.515327,203891.039161,2018.366254,22860.992562,4313.266226,4.968878e+05,...,-5.363541,17.209942,6.462601,21.442928,2.354765,24.843613,0.675824,12.787750,8,9


In [5]:
from sklearn.preprocessing import MinMaxScaler

X = df.drop(['res'], axis=1).to_numpy()
y = df['res'].to_numpy()

scaler = MinMaxScaler()
scaler.fit(X)
X_scal = scaler.transform(X)

In [6]:
round(X_scal[9][-1],1) == 1

np.True_

In [7]:
X_new = []
y_new = []

temp = []
for i in range(len(X_scal)-1):
    temp.append(X_scal[i])
    if round(X_scal[:,-1][i+1],1) == 0 and round(X_scal[:,-1][i],1) == 1:
        X_new.append(temp)
        y_new.append(y[i])
        temp = []
    elif round(X_scal[:,-1][i+1],1) == 0 and round(X_scal[:,-1][i],1) == 0.9:
        temp = []

X_new = np.array(X_new).astype('float32')
y_new = np.array(y_new).astype('float32')


In [8]:
x_train = []
x_test = []
y_train = []
y_test = []

for i in range(len(y_new)):
    if i%9 == 0:
        x_test.append(X_new[i])
        y_test.append(y_new[i])
    else:
        x_train.append(X_new[i])
        y_train.append(y_new[i])

x_train = np.array(x_train).astype('float32')
x_test = np.array(x_test).astype('float32')
y_train = np.array(y_train).astype('float32')
y_test = np.array(y_test).astype('float32')

In [9]:
x_train.shape

(879, 10, 58)

In [10]:
import collections
from collections import Counter

Counter(y_test)

Counter({np.float32(0.0): 12,
         np.float32(1.0): 10,
         np.float32(2.0): 11,
         np.float32(3.0): 11,
         np.float32(4.0): 11,
         np.float32(5.0): 11,
         np.float32(6.0): 11,
         np.float32(7.0): 11,
         np.float32(8.0): 12,
         np.float32(9.0): 10})

In [11]:
from sklearn.preprocessing import StandardScaler
orig_shape = x_train.shape
x_train_2d = x_train.reshape(-1, orig_shape[-1])
scaler = StandardScaler().fit(x_train_2d)
x_train = scaler.transform(x_train_2d).reshape(orig_shape)
x_test = scaler.transform(x_test.reshape(-1, orig_shape[-1])).reshape(x_test.shape)

In [12]:
def add_noise(x, noise_factor=0.05):
    noise = np.random.randn(*x.shape).astype(np.float32) * noise_factor
    return x + noise

In [13]:
x_train_aug = np.concatenate([x_train, add_noise(x_train)], axis=0)
y_train_aug = np.concatenate([y_train, y_train], axis=0)

In [14]:
!pip install tensorflow

In [46]:
import tensorflow as tf
import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

model = tf.keras.Sequential([
    tf.keras.layers.GRU(64, activation='tanh', return_sequences=True, recurrent_dropout=0.3, kernel_regularizer=regularizers.l2(0.0005),
                        input_shape=(x_train_aug.shape[1], x_train_aug.shape[2])),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.GRU(64, activation='tanh', return_sequences=True),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.GRU(32, activation='tanh'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),#3e-4
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_accuracy',
                    patience = 50,
                    min_delta = 0.001,
                    verbose = 0,
                    restore_best_weights = True,
                    mode = 'max'
                    )

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=10,
    min_lr=1e-5, verbose=1
)

model.fit(
        x = x_train_aug,
        y = y_train_aug,
        batch_size = 64,
        epochs = 500,
        verbose = 0,
        callbacks = [early_stopping, reduce_lr],
        shuffle = True,
        validation_split=0.15,
        validation_data = None,
        validation_batch_size = 16
        )

train_loss, train_acc = model.evaluate(x_train_aug, y_train_aug, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nAccuracy train: {train_acc:.4f}, test: {test_acc:.4f}")

Версия TensorFlow: 2.20.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True

Epoch 11: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 21: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 200: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.

Epoch 215: ReduceLROnPlateau reducing learning rate to 1e-05.

Accuracy train: 0.8527, test: 0.7545


In [59]:
x_train_2d = x_train_aug[..., np.newaxis]
x_test_2d = x_test[..., np.newaxis]

In [60]:
from tensorflow.keras import layers, models, callbacks, regularizers

model = models.Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0003),
                  input_shape=(x_train_aug.shape[1], x_train_aug.shape[2])),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.4),

    layers.Conv1D(128, kernel_size=3, activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0005)),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.6),

    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=50,
                                     restore_best_weights=True, mode='max')
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                        patience=12, min_lr=1e-5, verbose=1)

model.fit(
    x_train_aug, y_train_aug,
    batch_size=32,
    epochs=500,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    shuffle=True, verbose=0
)

train_loss, train_acc = model.evaluate(x_train_aug, y_train_aug, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Train accuracy: {train_acc:.4f}, Test accuracy: {test_acc:.4f}")


Epoch 105: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.

Epoch 117: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.

Epoch 129: ReduceLROnPlateau reducing learning rate to 3.7500001781154424e-05.

Epoch 141: ReduceLROnPlateau reducing learning rate to 1.8750000890577212e-05.
Train accuracy: 0.9983, Test accuracy: 0.7727


In [64]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

model = models.Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001),
                  input_shape=(x_train_aug.shape[1], x_train_aug.shape[2])),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.Conv1D(128, kernel_size=3, activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.GRU(128, activation='tanh', return_sequences=False,
               kernel_regularizer=regularizers.l2(0.0005),
               recurrent_dropout=0.2),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=50,
                                     restore_best_weights=True, mode='max')
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                        patience=12, min_lr=1e-5, verbose=1)

history = model.fit(
    x_train_aug, y_train_aug,
    batch_size=32, epochs=500,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    shuffle=True, verbose=0
)

train_loss, train_acc = model.evaluate(x_train_aug, y_train_aug, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Train accuracy: {train_acc:.4f}, Test accuracy: {test_acc:.4f}")


Epoch 65: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 80: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 92: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 104: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 116: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
Train accuracy: 0.9989, Test accuracy: 0.7727


In [67]:
inputs = layers.Input(shape=(x_train_aug.shape[1], x_train_aug.shape[2]))

x = layers.GRU(64, activation='tanh', return_sequences=True,
               kernel_regularizer=regularizers.l2(0.0005),
               recurrent_dropout=0.3)(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

x = layers.GRU(64, activation='tanh', return_sequences=True,
               kernel_regularizer=regularizers.l2(0.0005),
               recurrent_dropout=0.3)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

x = layers.GRU(32, activation='tanh', return_sequences=True,
               kernel_regularizer=regularizers.l2(0.0005))(x)
attention_out = layers.Attention()([x, x])
context = layers.GlobalAveragePooling1D()(attention_out)


x = layers.Dropout(0.5)(context)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_accuracy', patience=50, min_delta=0.001,
    restore_best_weights=True, mode='max', verbose=0
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=12, min_lr=1e-5, verbose=1
)

history = model.fit(
    x_train_aug, y_train_aug,
    batch_size=32, epochs=500,
    validation_split=0.15,
    callbacks=[early_stopping, reduce_lr],
    shuffle=True, verbose=0
)

train_loss, train_acc = model.evaluate(x_train_aug, y_train_aug, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Train accuracy: {train_acc:.4f}, Test accuracy: {test_acc:.4f}")


Epoch 218: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 230: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 242: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.

Epoch 254: ReduceLROnPlateau reducing learning rate to 1e-05.
Train accuracy: 0.9824, Test accuracy: 0.7818
